# PAF Experiments on Tabular Data

Pilot study extending *On Embeddings for Numerical Features in Tabular Deep Learning* (Gorishniy et al., NeurIPS 2022).

**Structure (mirrors original paper):**
1. Each model is first evaluated with **default HPs** (no tuning)
2. Then evaluated with **tuned HPs** (Optuna TPE, val R² as objective)
3. Final table shows both columns side by side — same format as Table 6 in the original paper

## Setup

In [1]:
import os, sys, json

PROJECT_ROOT = '/kaggle/input/paf-experiments'
if os.path.exists(PROJECT_ROOT):
    sys.path.insert(0, PROJECT_ROOT)
else:
    sys.path.insert(0, '/kaggle/working')

DATA_ROOT    = '/home/artem/Articles_and_Projects/Catboost_RT_vs_TabM/datasets_original' # '/kaggle/input/datasets-in-tabm'
RESULTS_DIR  = '/home/artem/Articles_and_Projects/Advanced_Embeddings_for_Tab_Data/results'

import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch: 2.10.0+cu128
CUDA: False


In [2]:
!pip install optuna -q

In [3]:
from data.loader          import load_dataset
from experiments.runner   import run_experiments, DefaultHParams, ALL_VARIANTS
from experiments.tuner    import tune_all, build_model_from_hp
from experiments.trainer  import train
from results.analysis     import aggregate, print_table

print('Variants:', len(ALL_VARIANTS))
for v in ALL_VARIANTS:
    print(' ', v)

Variants: 13
  MLP
  EmbMLP-orig-plain
  EmbMLP-orig-sc
  EmbMLP-orig-sc_af
  EmbMLP-grid-plain
  EmbMLP-grid-sc
  EmbMLP-grid-sc_af
  PAFNet-plain-ln
  PAFNet-sc-ln
  PAFNet-sc_af-ln
  PAFNet-plain-noln
  PAFNet-sc-noln
  PAFNet-sc_af-noln


/home/artem/.pyenv/versions/3.12.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

In [4]:
DATASETS  = ['california']   # add more as needed: 'house', 'adult', etc.
N_SEEDS   = 3                # seeds for final evaluation
N_TRIALS  = 50               # Optuna trials per variant
N_EPOCHS  = 100
PATIENCE  = 16

# Default HPs (no tuning)
default_hp = DefaultHParams(
    hidden_dim=64, n_layers=2, k=16, dropout=0.1,
    lr=1e-3, weight_decay=1e-5,
    n_epochs=N_EPOCHS, patience=PATIENCE,
    batch_size=256, n_seeds=N_SEEDS,
)

## Load datasets

In [5]:
datasets = {}
for ds_name in DATASETS:
    datasets[ds_name] = load_dataset(ds_name, data_root=DATA_ROOT, batch_size=256)
    print(f"  {ds_name}: n_features={datasets[ds_name]['n_features']}")

  [california] n_features=8 (num=8, bin=0, cat_ohe=0, cat_num=0)
  california: n_features=8


## Part 1 — Default HPs

Run all variants with fixed hyperparameters, no tuning.
Quick baseline to see which directions are promising.

In [6]:
results_default = run_experiments(
    dataset_names=DATASETS,
    data_root=DATA_ROOT,
    results_dir=RESULTS_DIR + '/default',
    variants=ALL_VARIANTS,
    hp=default_hp,
    device=device,
    verbose=True,
    do_tune=False,
)


  Dataset: california
  [california] n_features=8 (num=8, bin=0, cat_ohe=0, cat_num=0)
  n_features=8  task=regression  n_classes=1

  [MLP]  seed=42


/home/artem/.pyenv/versions/3.12.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


  [   1] train_loss=0.6146  val_loss=0.3345  val_R2=0.6544  *
  [  10] train_loss=0.2740  val_loss=0.2417  val_R2=0.7503  *
  [  20] train_loss=0.2514  val_loss=0.2274  val_R2=0.7651  *
  [  30] train_loss=0.2391  val_loss=0.2224  val_R2=0.7702
  [  40] train_loss=0.2276  val_loss=0.2150  val_R2=0.7779  *
  [  50] train_loss=0.2216  val_loss=0.2075  val_R2=0.7857  *
  [  60] train_loss=0.2144  val_loss=0.2047  val_R2=0.7885
  [  70] train_loss=0.2073  val_loss=0.1985  val_R2=0.7950  *
  [  80] train_loss=0.2015  val_loss=0.2016  val_R2=0.7918
  [  90] train_loss=0.1984  val_loss=0.1957  val_R2=0.7979
  [ 100] train_loss=0.1948  val_loss=0.1958  val_R2=0.7977
  → best_epoch=99  best_val_R2=0.8019  test_R2=0.7966  (16.8s)

  [MLP]  seed=43
  [   1] train_loss=0.6260  val_loss=0.3194  val_R2=0.6700  *
  [  10] train_loss=0.2716  val_loss=0.2398  val_R2=0.7522  *
  [  20] train_loss=0.2499  val_loss=0.2282  val_R2=0.7643  *
  [  30] train_loss=0.2362  val_loss=0.2224  val_R2=0.7703
  [  40

## Part 2 — Tuned HPs (Optuna TPE)

For each variant: run N_TRIALS Optuna trials optimising val R²,
then re-evaluate best config with N_SEEDS seeds.

In [ ]:
import statistics, math
from experiments.tuner  import tune, build_model_from_hp
from experiments.runner import run_one
from pathlib import Path

results_tuned = []
tuning_info   = {}   # {ds_name: {model_name: tune_result}}

for ds_name in DATASETS:
    dataset = datasets[ds_name]
    tuning_info[ds_name] = {}
    print(f"\n{'='*60}\n  Tuning on: {ds_name}\n{'='*60}")

    for model_name in ALL_VARIANTS:
        if 'PAFNet-plain' in model_name:
            continue
        
        print(f"\n  [HPO] {model_name} — {N_TRIALS} trials...")
        t = tune(
            model_name=model_name,
            dataset=dataset,
            n_trials=N_TRIALS,
            device=device,
            seed=42,
            n_epochs=N_EPOCHS,
            patience=PATIENCE,
            show_progress=True,
        )
        tuning_info[ds_name][model_name] = t
        print(f"  [HPO] best_val_R2={t['best_val_r2']:.4f}  "
              f"hp={t['best_hp']}  lr={t['best_lr']:.2e}")

        # Evaluate best HP with N_SEEDS seeds
        seed_results = []
        for seed_idx in range(N_SEEDS):
            actual_seed = 42 + seed_idx
            print(f"  [{model_name}]  seed={actual_seed}  (tuned)")
            r = run_one(
                model_name=model_name,
                dataset=dataset,
                hp=default_hp,
                results_dir=Path(RESULTS_DIR + '/tuned'),
                device=device,
                seed=actual_seed,
                verbose=True,
                tuned_hp=t['best_hp'],
                tuned_lr=t['best_lr'],
                tuned_wd=t['best_wd'],
            )
            seed_results.append(r)
            results_tuned.append(r)

        vals  = [r['best_val_metric'] for r in seed_results]
        tests = [r['test_metric']     for r in seed_results]
        std_v = statistics.stdev(vals)  if len(vals)  > 1 else 0.0
        std_t = statistics.stdev(tests) if len(tests) > 1 else 0.0
        print(f"  → {model_name:30s} "
              f"val_R2={statistics.mean(vals):.4f}±{std_v:.4f}  "
              f"test_R2={statistics.mean(tests):.4f}±{std_t:.4f}")


  Tuning on: california

  [HPO] MLP — 50 trials...


  0%|          | 0/50 [00:00<?, ?it/s]/home/artem/.pyenv/versions/3.12.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Best trial: 18. Best value: 0.814282:  42%|████▏     | 21/50 [26:56<42:00, 86.92s/it]   

## Results: Default HPs vs Tuned HPs

Side-by-side comparison, test R² (mean ± std over seeds).

In [ ]:
import pandas as pd

def _agg(results, ds_name):
    """Return {model_name: (test_mean, test_std)} for one dataset."""
    from collections import defaultdict
    groups = defaultdict(list)
    for r in results:
        if r['dataset_name'] == ds_name:
            groups[r['model_name']].append(r['test_metric'])
    out = {}
    for model, vals in groups.items():
        out[model] = (
            statistics.mean(vals),
            statistics.stdev(vals) if len(vals) > 1 else 0.0,
        )
    return out


for ds_name in DATASETS:
    agg_def   = _agg(results_default, ds_name)
    agg_tuned = _agg(results_tuned,   ds_name)

    rows = []
    baseline_def   = agg_def.get('MLP',   (None, None))[0]
    baseline_tuned = agg_tuned.get('MLP', (None, None))[0]

    for model_name in ALL_VARIANTS:
        d = agg_def.get(model_name)
        t = agg_tuned.get(model_name)

        def _fmt(tup):
            if tup is None: return '—'
            return f'{tup[0]:.4f} ±{tup[1]:.4f}'

        def _delta(tup, baseline):
            if tup is None or baseline is None: return ''
            if model_name == 'MLP': return ''
            delta = tup[0] - baseline
            arrow = '▲' if delta > 0 else '▼'
            return f'{arrow}{abs(delta):.4f}'

        rows.append({
            'Model':          model_name,
            'Default Test R²': _fmt(d),
            'Δ (vs MLP)':     _delta(d, baseline_def),
            'Tuned Test R²':  _fmt(t),
            'Δ (vs MLP)  ':   _delta(t, baseline_tuned),
        })

    df = pd.DataFrame(rows).set_index('Model')
    print(f'\nDataset: {ds_name}  (Test R² ↑, mean ± std over {N_SEEDS} seeds)\n')
    display(df)

## Save tuning info

In [ ]:
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Save tuning results (without optuna Study objects)
tuning_slim = {
    ds: {
        model: {'best_hp': t['best_hp'], 'best_lr': t['best_lr'],
                'best_wd': t['best_wd'], 'best_val_r2': t['best_val_r2']}
        for model, t in models.items()
    }
    for ds, models in tuning_info.items()
}
with open(RESULTS_DIR + '/tuning_info.json', 'w') as f:
    json.dump(tuning_slim, f, indent=2)

# Save evaluation results
all_results = results_default + results_tuned
with open(RESULTS_DIR + '/all_results.json', 'w') as f:
    json.dump(
        [{k: v for k, v in r.items() if k != 'history'} for r in all_results],
        f, indent=2
    )

print('Saved to', RESULTS_DIR)

## Learning curves

In [ ]:
import matplotlib.pyplot as plt

PLOT_VARIANTS = ['MLP', 'EmbMLP-grid-sc_af', 'PAFNet-sc_af-ln']
DATASET       = DATASETS[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (results, title) in zip(axes, [
    (results_default, 'Default HPs'),
    (results_tuned,   'Tuned HPs'),
]):
    for r in results:
        if (r['dataset_name'] == DATASET
                and r['model_name'] in PLOT_VARIANTS
                and r['seed'] == 42
                and 'history' in r):
            epochs = [h['epoch']      for h in r['history']]
            vals   = [h['val_metric'] for h in r['history']]
            ax.plot(epochs, vals, label=r['model_name'])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val R²')
    ax.set_title(f'{DATASET} — {title}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR + '/learning_curves.png', dpi=150)
plt.show()